# Motion coherence discrimination (Britten et al., 1992): analysis

Computes the paper's measures from one or more run folders (each holding a `*_trials.csv`):

1. **Psychometric function** per run (a subject's session): proportion correct against motion correlation, fitted by maximum likelihood with the Weibull form of Eq. 1, $p = 1 - 0.5\,e^{-(c/\alpha)^\beta}$ (p. 4749). $\alpha$ is the threshold (82 % correct), $\beta$ the slope; Eq. 2 frees the asymptote when that fits significantly better (p. 4750).
2. **Neurometric function** (pp. 4750-4753), only when the trials table carries each trial's spike count over the 2-s viewing interval (`spike_count`, added after aligning a recording): ROC area of preferred- vs null-direction counts per level, its Weibull fit, the neuron/behaviour threshold ratio and the Eq. 3 test of one curve against two.

Only completed choices outside correction trials are analysed, as in the paper. At 0 % the "correct" answer is the trial's nominal direction, so that point is chance by construction.

Set `ANALYSIS_RUN` to a run folder (several separated by the OS path separator), or leave it unset to use the newest run under `data/` or `data-rehearsal/`. These numbers come from whatever runs you give it; a simulated run's numbers describe the simulated subject, not a replication.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from britten1992.analysis import (
    THRESHOLD_P,
    compare_fits,
    fit_psychometric,
    measured,
    neurometric_fit,
    neurometric_table,
    psychometric_table,
)

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUNS = os.environ.get('ANALYSIS_RUN')
if RUNS:
    RUNS = [Path(p) for p in RUNS.split(os.pathsep) if p]
else:
    found = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUNS = [found[-1].parent] if found else []
print('runs:', *RUNS, sep='\n  ')

In [ ]:
# 'anti' is the paper's null direction; read the table without turning words into NaN.
frames = []
for run in RUNS:
    for f in sorted(Path(run).glob('*_trials.csv')):
        df = pd.read_csv(f, keep_default_na=False, na_values=[''])
        df['run'] = Path(run).name
        df['session'] = Path(run).parent.name
        df['subject'] = Path(run).parent.parent.name
        frames.append(df)
trials = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(trials):
    trials['correction'] = trials['correction'].astype(str).str.lower().eq('true')
    print(trials.groupby(['subject', 'session', 'outcome']).size().unstack(fill_value=0))
    print('correction trials (excluded):', int(trials['correction'].sum()))

## Psychometric functions (Eq. 1 / Eq. 2)

In [ ]:
fits = {}
for key, g in trials.groupby(['subject', 'session', 'run']) if len(trials) else []:
    table = psychometric_table(g)
    if not len(table):
        continue
    fit = fit_psychometric(table)
    fits[key] = (table, fit)
    print(f'{key}: alpha = {fit.alpha:.2f} % correlation (threshold, {THRESHOLD_P:.0%} correct), '
          f'beta = {fit.beta:.2f}, delta = {fit.delta:.3f} (Eq. {fit.equation}), '
          f'{fit.n_trials} trials')
    print(table.to_string(index=False))

In [ ]:
if fits:
    fig, axes = plt.subplots(1, len(fits), figsize=(4.2 * len(fits), 3.4), squeeze=False)
    for ax, (key, (table, fit)) in zip(axes[0], fits.items(), strict=False):
        nonzero = table[table['coherence_pct'] > 0]
        ax.semilogx(nonzero['coherence_pct'], nonzero['p_correct'], 'o', color='k')
        c = np.geomspace(max(0.1, nonzero['coherence_pct'].min() / 2), 100, 200)
        ax.semilogx(c, fit.predict(c), '-', color='k')
        ax.axhline(0.5, color='0.7', lw=0.8)
        ax.set_ylim(0.4, 1.02)
        ax.set_xlabel('Correlation (%)')
        ax.set_ylabel('Proportion correct')
        ax.set_title(f'{key[0]} {key[1]}: alpha {fit.alpha:.1f} %, beta {fit.beta:.2f}', fontsize=9)
    fig.tight_layout()

## Neurometric functions (needs an MT recording)

A neurometric function needs each trial's spike count over the 2-s viewing interval, from a recording aligned to the session through the sync pulses (STIM_ON opens the window, TARGETS_ON closes it). Nothing here invents one: without a `spike_count` column the section reports that and stops.

In [ ]:
if len(trials) and 'spike_count' in trials:
    for key, g in trials.groupby(['subject', 'session', 'run']):
        neuro = neurometric_table(g)
        nfit = neurometric_fit(neuro)
        table, pfit = fits[key]
        n_half = ((neuro['n_pref'] + neuro['n_anti']) / 2).round()
        merged = table.merge(neuro, on='coherence_pct', suffixes=('', '_neuro'))
        lam, p = compare_fits(merged['coherence_pct'], merged['n_correct'], merged['n'],
                              (merged['p_correct_neuro'] * n_half).round(), n_half)
        print(f'{key}: neurometric alpha {nfit.alpha:.2f} %, beta {nfit.beta:.2f}; '
              f'threshold ratio neuron/behaviour {nfit.alpha / pfit.alpha:.2f}; '
              f'Eq. 3 lambda {lam:.2f}, p {p:.3f}')
else:
    print('no spike counts in these runs: the neurometric half needs a recording '
          '(docs/methods.md, What a recording rig adds)')

## Saccade reports (monkey set)

Saccade latency from target onset, by correlation; the paper reports no latencies, so this is a data check, not a replication measure.

In [ ]:
if len(trials) and 'saccade_rt_ms' in trials:
    m = measured(trials)
    m = m[m['saccade_rt_ms'].notna()]
    if len(m):
        print(m.groupby('coherence_pct')['saccade_rt_ms'].describe()[['count', 'mean', '50%']])
else:
    print('no saccade reports in these runs (keyboard set)')